In [1]:
# Implement K-Nearest Neighbors algorithm on diabetes.csv dataset. Compute confusion
# matrix, accuracy, error rate, precision and recall on the given dataset.
# Dataset link : https://www.kaggle.com/datasets/abdallamahgoub/diabetes

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("diabetes.csv")

In [5]:
df.sample(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Pedigree,Age,Outcome
191,9,123,70,44,94,33.1,0.374,40,0
11,10,168,74,0,0,38.0,0.537,34,1


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Pregnancies    768 non-null    int64  
 1   Glucose        768 non-null    int64  
 2   BloodPressure  768 non-null    int64  
 3   SkinThickness  768 non-null    int64  
 4   Insulin        768 non-null    int64  
 5   BMI            768 non-null    float64
 6   Pedigree       768 non-null    float64
 7   Age            768 non-null    int64  
 8   Outcome        768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [7]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Pedigree,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [8]:
import numpy as np

# 1. List the columns where 0 is biologically impossible
cols_with_hidden_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# 2. Replace the 0s with NaNs
df[cols_with_hidden_missing] = df[cols_with_hidden_missing].replace(0, np.nan)

# 3. Check how many missing values you actually have now
print(df.isnull().sum())

Pregnancies        0
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
Pedigree           0
Age                0
Outcome            0
dtype: int64


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore') # Keeps output clean

# Define the columns that had hidden missing values
missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Create a master list to store our results
results = []

def evaluate_pipeline(name, X, y):
    # 1. Split Data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 2. Scale Data (KNN Requires Scaling)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 3. Train KNN Classifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)
    
    # 4. Evaluate
    preds = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results.append({'Method': name, 'Accuracy': acc})
    print(f"--- {name} ---")
    print(classification_report(y_test, preds))

# ==========================================
# Method 1: Drop Heavily Missing Columns & Drop Remaining NaN Rows
# ==========================================
df_drop = df.copy()
# Drop columns with too many NaNs
df_drop = df_drop.drop(columns=['Insulin', 'SkinThickness'])
# Drop the few remaining rows with NaNs (like the 5 missing Glucose)
df_drop = df_drop.dropna()

X1 = df_drop.drop(columns=['Outcome'])
y1 = df_drop['Outcome']
evaluate_pipeline("Method 1: Drop Cols/Rows", X1, y1)

# ==========================================
# Method 2: Grouped Median Imputation
# ==========================================
df_median = df.copy()
for col in missing_cols:
    df_median[col] = df_median.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

X2 = df_median.drop(columns=['Outcome'])
y2 = df_median['Outcome']
evaluate_pipeline("Method 2: Grouped Median", X2, y2)

# ==========================================
# Method 3: KNN Imputation
# ==========================================
df_knn_impute = df.copy()
# Note: Imputation should ideally happen *after* splitting, but for this quick 
# comparison script across the whole dataframe, we'll do it before to keep it simple.
# In production, fit the imputer on X_train only.
X_raw = df_knn_impute.drop(columns=['Outcome'])
y_raw = df_knn_impute['Outcome']

imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)

evaluate_pipeline("Method 3: KNN Imputation", X_imputed, y_raw)

# ==========================================
# Final Results
# ==========================================
print("\n=== Final Accuracy Comparison ===")
for r in results:
    print(f"{r['Method']}: {r['Accuracy']:.4f}")

--- Method 1: Drop Cols/Rows ---
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       102
           1       0.55      0.60      0.58        43

    accuracy                           0.74       145
   macro avg       0.69      0.70      0.69       145
weighted avg       0.75      0.74      0.74       145

--- Method 2: Grouped Median ---
              precision    recall  f1-score   support

           0       0.87      0.81      0.84        99
           1       0.69      0.78      0.74        55

    accuracy                           0.80       154
   macro avg       0.78      0.79      0.79       154
weighted avg       0.81      0.80      0.80       154

--- Method 3: KNN Imputation ---
              precision    recall  f1-score   support

           0       0.80      0.75      0.77        99
           1       0.59      0.65      0.62        55

    accuracy                           0.71       154
   macro avg       0.69      

In [10]:
# the second method introduces data leakage as we take a grouped median based on outcome, which sometimes might be misleading, 
# or in short we train the model to learn medians for the particular outcome

In [11]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# 1. Prepare Method 1 Data (Clean and strict)
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Insulin', 'SkinThickness']).dropna()

X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Build a Pipeline (Scales the data dynamically to prevent leakage)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# 3. Define the Grid of Hyperparameters to test
param_grid = {
    # n_neighbors: Odd numbers prevent tie-votes between classes
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 19, 21, 25], 
    
    # weights: 'uniform' treats all neighbors equally, 'distance' gives closer neighbors more voting power
    'knn__weights': ['uniform', 'distance'],
    
    # metric: 'euclidean' is straight-line distance, 'manhattan' is grid-like distance
    'knn__metric': ['euclidean', 'manhattan']
}

# 4. Run the Grid Search (Tests all combinations using 5-Fold Cross Validation)
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
print("Tuning hyperparameters... This will test", len(param_grid['knn__n_neighbors']) * 2 * 2, "combinations.")
grid_search.fit(X_train, y_train)

# 5. Print the winning results
print("\n--- Tuning Complete ---")
print(f"Best Parameters Found: {grid_search.best_params_}")

# Evaluate the winning model on our completely unseen test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Final Tuned Performance ---")
print(classification_report(y_test, y_pred))

Tuning hyperparameters... This will test 36 combinations.

--- Tuning Complete ---
Best Parameters Found: {'knn__metric': 'manhattan', 'knn__n_neighbors': 7, 'knn__weights': 'uniform'}

--- Final Tuned Performance ---
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       102
           1       0.60      0.60      0.60        43

    accuracy                           0.77       145
   macro avg       0.72      0.72      0.72       145
weighted avg       0.77      0.77      0.77       145

